# Prophet Modeling for TEPCO Demand

## 1. Import Libraries

In [ ]:
from prophet import Prophet
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/processed/final_dataset.csv')
df['Datetime'] = pd.to_datetime(df['Datetime'])

# Rename for Prophet
pdf = df[['Datetime', 'Demand', 'Temperature']].rename(columns={'Datetime': 'ds', 'Demand': 'y'})
pdf.head()

## 3. Split Train/Test

In [ ]:
split_date = '2024-01-01'
train = pdf[pdf['ds'] < split_date]
test = pdf[pdf['ds'] >= split_date]
print(f"Train: {train.shape}, Test: {test.shape}")

## 4. Train Model

In [ ]:
m = Prophet(daily_seasonality=True)
m.add_country_holidays(country_name='JP')
m.add_regressor('Temperature')
m.fit(train)

## 5. Forecast

In [ ]:
future = test[['ds', 'Temperature']]
forecast = m.predict(future)
m.plot(forecast)
plt.show()

## 6. Evaluate

In [ ]:
y_true = test['y'].values
y_pred = forecast['yhat'].values.astype(float)
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
print(f"MAPE: {mape:.2f}%")